# Практика 1. Две витрины из одного источника

Стартовый ноутбук: выполняйте ячейки сверху вниз и заполняйте TODO. В этой версии часть SQL-запросов нужно дописать самостоятельно; после каждого пропуска указано, какой результат нужен. Подсказки по операциям есть в [плане практики](Практика_1_план.md), а готовое решение преподаватель выдаст после занятия.

После `docker compose up -d` PostgreSQL уже содержит учебные данные. Сначала строим первую витрину в pandas, затем сохраняем исходные строки в Parquet в MinIO и строим вторую витрину запросом ClickHouse. В конце сравниваем обе витрины. Ноутбук запускается на компьютере, не в Docker.


## 1. Окружение и подключения

Если VS Code попросит перезапустить kernel после установки пакетов, сделайте это.


In [1]:
%pip install -q "psycopg[binary]" boto3 clickhouse-connect pandas pyarrow


Note: you may need to restart the kernel to use updated packages.


In [1]:
import io
from datetime import date

import boto3
from botocore.config import Config
import clickhouse_connect
import pandas as pd
import psycopg

PG = dict(host='localhost', port=5433, dbname='shop', user='student', password='studentpass')
CH = dict(host='localhost', port=8123, database='de_course', username='student', password='studentpass')
S3_ENDPOINT = 'http://localhost:9000'  # ноутбук → MinIO с компьютера
S3_USER = 'minioadmin'
S3_PASSWORD = 'minioadmin'
BUCKET = 'de-course'
KEY = f'raw/orders/load_date={date.today().isoformat()}/orders.parquet'
S3_URL_FOR_CLICKHOUSE = f'http://minio:9000/{BUCKET}/{KEY}'  # ClickHouse → MinIO внутри Docker


## 2. Общий источник — PostgreSQL

В DBeaver проверьте число заказов и распределение статусов. Затем допишите `SELECT` ниже: нужны `order_id`, дата заказа **как текст** под именем `order_date`, `status`, `amount_rub` и `campaign`. Таблицы `orders` и `marketing_attribution` уже соединены через `order_id`; `LEFT JOIN` сохраняет заказ даже без привязки к кампании. Сортировка по `order_id` нужна для воспроизводимого порядка строк.

Подсказка PostgreSQL: преобразование даты в текст записывается как `o.order_date::text AS order_date`. После заполнения удалите строку комментария `-- TODO`; проверка под ячейкой пропустит запрос дальше.


In [ ]:
sql = '''
SELECT
    -- TODO: перечислите пять нужных полей с префиксами o. и a.; дату приведите к text
FROM orders AS o
LEFT JOIN marketing_attribution AS a USING (order_id)
ORDER BY o.order_id
'''
if '-- TODO' in sql:
    raise NotImplementedError('Допишите SELECT для чтения PostgreSQL')

with psycopg.connect(**PG) as conn:
    with conn.cursor() as cur:
        cur.execute(sql)
        columns = [column.name for column in cur.description]
        records = [dict(zip(columns, row)) for row in cur.fetchall()]
assert len(records) == 120
print('Прочитано заказов:', len(records))
display(pd.DataFrame(records).head())


**Контрольная точка 1.** Почему общую сумму всех заказов нельзя назвать выручкой? **Ваш ответ:** …


## 3. Подход 1 — преобразовать в pandas, затем загрузить

Сводная таблица суммирует заказы по кампании и статусу. Оставляем `paid` и `returned`; столбец `pending` не берём. Для этой учебной выборки у каждой кампании есть оба нужных статуса. Преобразование происходит **до** загрузки в ClickHouse.


In [ ]:
source_df = pd.DataFrame(records)
totals = source_df.pivot_table(
    index='campaign', columns='status', values='amount_rub', aggfunc='sum'
).reset_index()
pandas_mart = totals[['campaign', 'paid', 'returned']].rename(
    columns={'paid': 'paid_rub', 'returned': 'returned_rub'}
)
# TODO: добавьте столбец net_rub как paid_rub минус returned_rub
raise NotImplementedError('Рассчитайте net_rub в pandas')
pandas_rows_to_load = pandas_mart[['campaign', 'paid_rub', 'returned_rub', 'net_rub']].values.tolist()
display(pandas_mart)


In [ ]:
ch = clickhouse_connect.get_client(**CH)
ch.command('''
CREATE TABLE IF NOT EXISTS de_course.campaign_revenue_pandas (
    campaign String, paid_rub Int64, returned_rub Int64, net_rub Int64
) ENGINE = MergeTree ORDER BY campaign
''')
ch.command('TRUNCATE TABLE de_course.campaign_revenue_pandas')
ch.insert(
    'de_course.campaign_revenue_pandas', pandas_rows_to_load,
    column_names=['campaign', 'paid_rub', 'returned_rub', 'net_rub'],
)
pandas_rows = ch.query('''
SELECT campaign, paid_rub, returned_rub, net_rub
FROM de_course.campaign_revenue_pandas ORDER BY campaign
''').result_rows
print('Подход 1:', pandas_rows)


**Контрольная точка 2.** Где именно выполнено преобразование данных в этой ветке? **Ваш ответ:** …


## 4. Подход 2 — загрузить исходные строки в Parquet

DataFrame ниже используется только для записи Parquet. Здесь ещё **нет** группировки или расчёта выручки. После загрузки объекта допишите короткий SQL-запрос проверки: ClickHouse должен посчитать число строк прямо в объекте MinIO через уже указанную функцию `s3(...)`. Это не подсчёт строк витрины.


In [ ]:
s3 = boto3.client(
    's3', endpoint_url=S3_ENDPOINT,
    aws_access_key_id=S3_USER, aws_secret_access_key=S3_PASSWORD,
    region_name='us-east-1',
    config=Config(signature_version='s3v4', s3={'addressing_style': 'path'}),
)
existing = {item['Name'] for item in s3.list_buckets()['Buckets']}
if BUCKET not in existing:
    s3.create_bucket(Bucket=BUCKET)
raw_df = pd.DataFrame(records)
buffer = io.BytesIO()
raw_df.to_parquet(buffer, engine='pyarrow', index=False)
s3.put_object(Bucket=BUCKET, Key=KEY, Body=buffer.getvalue(), ContentType='application/octet-stream',
              Metadata={'source': 'postgres', 'layer': 'raw'})
print(f's3://{BUCKET}/{KEY}', '-', buffer.tell(), 'байт')


In [ ]:
response = s3.head_object(Bucket=BUCKET, Key=KEY)
print('Объект найден; размер:', response['ContentLength'], 'байт')
raw_count_sql = f"""
SELECT
    -- TODO: посчитайте число исходных строк функцией count()
FROM s3('{S3_URL_FOR_CLICKHOUSE}', '{S3_USER}', '{S3_PASSWORD}', 'Parquet')
"""
if '-- TODO' in raw_count_sql:
    raise NotImplementedError('Допишите запрос подсчёта строк Parquet')
raw_count_in_ch = ch.query(raw_count_sql).result_rows[0][0]
assert raw_count_in_ch == len(records)
print('ClickHouse прочитал из Parquet:', raw_count_in_ch, 'строк')


**Контрольная точка 3.** Покажите объект в MinIO Console. Почему ноутбук использует `localhost:9000`, а ClickHouse — `minio:9000`? **Ваш ответ:** …


## 5. Подход 2 — преобразовать в ClickHouse

ClickHouse читает исходный Parquet из MinIO и сам считает агрегат в SQL. Допишите три выражения в запросе ниже: суммы отдельно для статусов `paid` и `returned` (через `sumIf`), затем `net_rub` как разность этих сумм. У каждого выражения должен быть псевдоним, соответствующий колонке витрины. Не забудьте запятые между выражениями `SELECT` и удалите все строки `-- TODO` после заполнения.

Таблица создаётся и очищается готовыми командами: на этом занятии мы концентрируемся на правилах расчёта и местах преобразования.


In [ ]:
ch.command('''
CREATE TABLE IF NOT EXISTS de_course.campaign_revenue_s3 (
    campaign String, paid_rub Int64, returned_rub Int64, net_rub Int64
) ENGINE = MergeTree ORDER BY campaign
''')
s3_sql = f"""
INSERT INTO de_course.campaign_revenue_s3
SELECT campaign, paid_rub, returned_rub,
       -- TODO: разность оплаченных сумм и возвратов AS net_rub
FROM (
    SELECT campaign,
           -- TODO: сумма amount_rub только для status = 'paid' AS paid_rub,
           -- TODO: сумма amount_rub только для status = 'returned' AS returned_rub
    FROM s3('{S3_URL_FOR_CLICKHOUSE}', '{S3_USER}', '{S3_PASSWORD}', 'Parquet')
    GROUP BY campaign
)
"""
if '-- TODO' in s3_sql:
    raise NotImplementedError('Допишите расчёт витрины в SQL ClickHouse')
ch.command('TRUNCATE TABLE de_course.campaign_revenue_s3')
ch.command(s3_sql)
s3_rows = ch.query('''
SELECT campaign, paid_rub, returned_rub, net_rub
FROM de_course.campaign_revenue_s3 ORDER BY campaign
''').result_rows
print('Подход 2:', s3_rows)


## 6. Сравнить две витрины

Повторно выполните целиком ячейки загрузки обоих подходов. После повторов результат должен остаться прежним.


In [ ]:
assert pandas_rows == s3_rows, 'Витрины различаются: сравните правила расчёта'
assert len(pandas_rows) == 4
print('Витрины идентичны:')
for row in pandas_rows:
    print(row)
print('Общая чистая сумма:', sum(row[3] for row in pandas_rows), 'руб.')


**Контрольная точка 4.** Чем различаются два подхода? Какой из них относится к ETL, а какой — к ELT? Почему обе витрины совпали и какие ограничения есть у схемы `TRUNCATE` + загрузка? **Ваш ответ:** …
